# Prompt Brittleness & the Illusion of "Same Metric"

**Same name. Same 1–5 scale. Same few-shot examples. Different rubric wording —
big score swings.**

The usual framing of brittleness is about punctuation and delimiters. On modern
models those barely move the needle (we'll keep `v1 Reversed examples` as a
presentation control to show this). The **real brittleness** is more insidious:

> Two rubrics with the same name, same scale, and seemingly synonymous wording are
> actually **different instruments**. The model follows each faithfully — and gives
> you different scores.

If you didn't realize you swapped the instrument, you'll attribute the score
change to the system. That's the bug — on our side, not the model's.

In [ ]:
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-4.1-mini"
CACHE_PATH = Path("brittleness_data_cache.json")

print("model:", MODEL)
print("cache:", "found" if CACHE_PATH.exists() else "will generate")

## Step 1 — Eval data (20 customer-support exchanges with ground-truth scores)

In [ ]:
GENERATE_PROMPT = """Generate a realistic customer support exchange for an e-commerce company.
Return JSON with:
- "query": customer message (1-3 sentences)
- "response": agent response (2-4 sentences)
- "true_score": Correctness score 1-5
- "reasoning": why this score (1 sentence)

Target quality level: {target_score}
Return ONLY valid JSON."""


def generate_eval_data(n=20):
    import random
    random.seed(123)
    targets = [random.choice([1, 2, 3, 3, 4, 4, 4, 5, 5]) for _ in range(n)]
    examples = []
    print(f"generating {n} examples...")
    for i, target in enumerate(targets):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": GENERATE_PROMPT.format(target_score=target)}],
            temperature=0.9,
            response_format={"type": "json_object"},
        )
        try:
            ex = json.loads(resp.choices[0].message.content)
            ex["id"] = i
            examples.append(ex)
        except json.JSONDecodeError:
            continue
    return examples


if CACHE_PATH.exists():
    eval_data = json.loads(CACHE_PATH.read_text())
    print(f"loaded cache: {len(eval_data)} examples")
else:
    eval_data = generate_eval_data(20)
    CACHE_PATH.write_text(json.dumps(eval_data, indent=2))
    print(f"generated and cached: {len(eval_data)} examples")

In [ ]:
# Peek
ex = eval_data[0]
print("query:    ", ex["query"])
print("response: ", ex["response"])
print("true:     ", ex["true_score"], "—", ex["reasoning"])

## Step 2 — Shared few-shot calibration examples

All variants use the **same** 3 examples. The only thing we vary is the rubric wording.

In [ ]:
FEWSHOT = [
    {
        "query": "Can I return this item after 45 days?",
        "response": "Our return policy allows returns within 30 days of purchase. "
                    "Unfortunately, after 45 days we cannot process a return, but I "
                    "can help you with an exchange or store credit.",
        "score": 4,
        "reasoning": "Correct policy, offers alternatives, minor gap on exceptions."
    },
    {
        "query": "Is this laptop compatible with Linux?",
        "response": "Yes! This model ships with Windows but has full Linux compatibility. "
                    "Ubuntu and Fedora are officially supported, and the WiFi and Bluetooth "
                    "drivers work out of the box.",
        "score": 5,
        "reasoning": "Fully correct, specific, addresses the exact question."
    },
    {
        "query": "Where is my order? I ordered it last week.",
        "response": "I apologize for the inconvenience! Let me look into that. Our "
                    "delivery team is working hard to get your order to you as soon "
                    "as possible.",
        "score": 2,
        "reasoning": "Doesn't actually check the order status or provide tracking. Vague deflection."
    },
]


def fewshot_block(examples, delimiter="---"):
    return "\n\n".join(
        f"{delimiter}\nCustomer: {fs['query']}\nAgent: {fs['response']}\n"
        f"Score: {fs['score']}\nReasoning: {fs['reasoning']}"
        for fs in examples
    )


FEWSHOT_BLOCK = fewshot_block(FEWSHOT)
print(FEWSHOT_BLOCK[:300], "...")

## Step 3 — Define the 8 variants

- **v0** is the reference rubric.
- **v1** is a **presentation control** — reversed few-shot order, same wording. We expect
  little movement; this is the "classic" brittleness test.
- **v2–v7** are **semantic perturbations**: "nuanced" rewordings that a PM would approve
  in review but that subtly change what the rubric measures.

In [ ]:
V0_BASELINE = f"""Rate the following customer support response for Correctness (1-5).

## Rubric
1 = Wrong or irrelevant
2 = Mostly wrong
3 = Partially correct, missing key details
4 = Mostly correct, minor gaps
5 = Fully correct and complete

## Examples
{FEWSHOT_BLOCK}

---
Customer: {{query}}
Agent: {{response}}

Return JSON with "score" (1-5) and "reasoning" (1-2 sentences). Return ONLY valid JSON."""


V1_REVERSED = f"""Rate the following customer support response for Correctness (1-5).

## Rubric
1 = Wrong or irrelevant
2 = Mostly wrong
3 = Partially correct, missing key details
4 = Mostly correct, minor gaps
5 = Fully correct and complete

## Examples
{fewshot_block(list(reversed(FEWSHOT)))}

---
Customer: {{query}}
Agent: {{response}}

Return JSON with "score" (1-5) and "reasoning" (1-2 sentences). Return ONLY valid JSON."""


V2_ACCURACY = f"""Rate the following customer support response for Accuracy (1-5).

## Rubric
1 = Inaccurate
2 = Mostly inaccurate
3 = Partially accurate
4 = Mostly accurate
5 = Fully accurate

## Examples
{FEWSHOT_BLOCK}

---
Customer: {{query}}
Agent: {{response}}

Return JSON with "score" (1-5) and "reasoning" (1-2 sentences). Return ONLY valid JSON."""


V3_STRICT = f"""Rate the following customer support response for Correctness (1-5).

**Score strictly.** Reserve 5 only for responses that are fully correct, complete,
and leave nothing ambiguous. Any missing detail, caveat, or step caps the score at 4.
When in doubt, score lower — the customer deserves an accurate bar.

## Rubric
1 = Wrong or irrelevant
2 = Mostly wrong
3 = Partially correct, missing key details
4 = Mostly correct, minor gaps
5 = Fully correct and complete

## Examples
{FEWSHOT_BLOCK}

---
Customer: {{query}}
Agent: {{response}}

Return JSON with "score" (1-5) and "reasoning" (1-2 sentences). Return ONLY valid JSON."""


V4_LENIENT = f"""Rate the following customer support response for Correctness (1-5).

**Score generously.** Support agents are trying their best under time pressure. Reward
responses that make a genuine effort to address the customer. A helpful-but-incomplete
answer should not be penalized harshly — partial credit is appropriate.

## Rubric
1 = Wrong or irrelevant
2 = Mostly wrong
3 = Partially correct, missing key details
4 = Mostly correct, minor gaps
5 = Fully correct and complete

## Examples
{FEWSHOT_BLOCK}

---
Customer: {{query}}
Agent: {{response}}

Return JSON with "score" (1-5) and "reasoning" (1-2 sentences). Return ONLY valid JSON."""


V5_CUSTOMER_PERSPECTIVE = f"""Rate this customer support exchange from the customer's
perspective. On a 1-5 scale, how satisfied would the customer be with this response?
Would they feel their concern was resolved?

## Rubric
1 = Very unsatisfied — the response did not help at all
2 = Unsatisfied — the response missed the point
3 = Neutral — the response was acceptable but underwhelming
4 = Satisfied — the response resolved the concern well
5 = Very satisfied — the response was everything the customer could ask for

## Examples
{FEWSHOT_BLOCK}

---
Customer: {{query}}
Agent: {{response}}

Return JSON with "score" (1-5) and "reasoning" (1-2 sentences). Return ONLY valid JSON."""


V6_GRADE_SCALE = f"""Rate the following customer support response for Correctness (1-5).

## Scale
1 = Unacceptable — would fail QA review
2 = Below expectations — needs rework
3 = Meets basic expectations — acceptable but not impressive
4 = Above expectations — good work
5 = Exceeds expectations — exceptional

## Examples
{FEWSHOT_BLOCK}

---
Customer: {{query}}
Agent: {{response}}

Return JSON with "score" (1-5) and "reasoning" (1-2 sentences). Return ONLY valid JSON."""


V7_MIDPOINT_SHIFT = f"""Rate the following customer support response for Correctness (1-5).

## Rubric
1 = Unusable — contains serious errors
2 = Needs major work — significant issues
3 = Acceptable baseline — solves the core issue
4 = Above baseline — solid, well-rounded
5 = Excellent — clearly outstanding

## Examples
{FEWSHOT_BLOCK}

---
Customer: {{query}}
Agent: {{response}}

Return JSON with "score" (1-5) and "reasoning" (1-2 sentences). Return ONLY valid JSON."""


VARIANTS = [
    ("v0 Baseline",             V0_BASELINE,             "reference"),
    ("v1 Reversed examples",    V1_REVERSED,             "presentation"),
    ("v2 Correctness→Accuracy", V2_ACCURACY,             "semantic"),
    ("v3 Strict framing",       V3_STRICT,               "semantic"),
    ("v4 Lenient framing",      V4_LENIENT,              "semantic"),
    ("v5 Customer perspective", V5_CUSTOMER_PERSPECTIVE, "semantic"),
    ("v6 Grade-like scale",     V6_GRADE_SCALE,          "semantic"),
    ("v7 Midpoint shift",       V7_MIDPOINT_SHIFT,       "semantic"),
]

for n, _, k in VARIANTS:
    print(f"  [{k:<12}] {n}")

## Step 4 — Run all variants on the same 20 examples

`temperature=0`. Only the prompt wording differs. ~3 minutes.

In [ ]:
def run_variant(name, prompt_template, data):
    results = []
    for ex in data:
        prompt = prompt_template.format(query=ex["query"], response=ex["response"])
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        try:
            parsed = json.loads(resp.choices[0].message.content)
            score = int(parsed.get("score", 0))
        except (json.JSONDecodeError, ValueError):
            score = 0
        results.append({"id": ex["id"], "true": ex["true_score"], "judge": score,
                        "exact": score == ex["true_score"],
                        "within1": abs(score - ex["true_score"]) <= 1})
    exact = sum(r["exact"] for r in results) / len(results) * 100
    within1 = sum(r["within1"] for r in results) / len(results) * 100
    mae = sum(abs(r["judge"] - r["true"]) for r in results) / len(results)
    mean_score = sum(r["judge"] for r in results) / len(results)
    return {"name": name, "exact": exact, "within1": within1, "mae": mae,
            "mean_score": mean_score, "results": results}


all_results = []
for name, template, kind in VARIANTS:
    print(f"running: {name}...")
    start = time.time()
    res = run_variant(name, template, eval_data)
    res["elapsed"] = time.time() - start
    res["kind"] = kind
    all_results.append(res)
    print(f"  exact={res['exact']:.1f}%  mean={res['mean_score']:.2f}  MAE={res['mae']:.2f}  ({res['elapsed']:.1f}s)")

## Step 5 — Summary table + swings

In [ ]:
baseline_mean = all_results[0]["mean_score"]
print(f"{'Variant':<28} {'Kind':<13} {'Exact%':>7} {'±1%':>6} {'MAE':>5} {'Mean':>5} {'Δ':>6}")
print("-" * 76)
for r in all_results:
    d = r["mean_score"] - baseline_mean
    print(f"{r['name']:<28} {r['kind']:<13} {r['exact']:>6.1f}% {r['within1']:>5.1f}% "
          f"{r['mae']:>4.2f} {r['mean_score']:>5.2f} {d:>+6.2f}")
print("-" * 76)

means = [r["mean_score"] for r in all_results]
exacts = [r["exact"] for r in all_results]
print(f"\nMean-score swing:  {max(means) - min(means):.2f}pts (on a 1–5 scale)")
print(f"Exact-match swing: {max(exacts) - min(exacts):.1f}pp")

## Step 6 — Charts

### a) Mean score per variant

The presentation control (v1) barely moves. Semantic perturbations pull the mean up
and down, because they are literally different instruments.

In [ ]:
names = [r["name"] for r in all_results]
means = [r["mean_score"] for r in all_results]
kinds = [r["kind"] for r in all_results]
color_map = {"reference": "#111827", "presentation": "#2563eb", "semantic": "#dc2626"}
colors = [color_map[k] for k in kinds]
baseline = means[0]

fig, ax = plt.subplots(figsize=(11, 4.5))
bars = ax.bar(names, means, color=colors)
ax.axhline(baseline, color="#111827", linestyle="--", linewidth=1, alpha=0.5, label=f"baseline = {baseline:.2f}")
ax.set_ylabel("Mean judge score (1–5)")
ax.set_ylim(0, 5)
ax.set_title("Same data, same model — mean score by rubric wording")
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
            f"{m:.2f}", ha="center", fontsize=10)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#111827", label="reference"),
    Patch(color="#2563eb", label="presentation perturbation"),
    Patch(color="#dc2626", label="semantic perturbation"),
], loc="lower right")
plt.setp(ax.xaxis.get_majorticklabels(), rotation=18, ha="right")
plt.tight_layout()
plt.show()

### b) Δmean vs baseline — sorted by magnitude

Presentation perturbations stay near zero. Semantic perturbations move visibly,
and one (customer perspective) moves **dramatically**.

In [ ]:
deltas = [(r["name"], r["kind"], r["mean_score"] - baseline) for r in all_results[1:]]  # skip baseline
deltas.sort(key=lambda x: abs(x[2]), reverse=True)

fig, ax = plt.subplots(figsize=(10, 4))
labels = [d[0] for d in deltas]
values = [d[2] for d in deltas]
cols = [color_map[d[1]] for d in deltas]
bars = ax.barh(labels, values, color=cols)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Δ mean score vs baseline (pts on 1–5 scale)")
ax.set_title("Nuanced rubric rewordings move the mean — sometimes a lot")
for bar, v in zip(bars, values):
    ax.text(v + (0.02 if v >= 0 else -0.02), bar.get_y() + bar.get_height()/2,
            f"{v:+.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### c) Per-example heatmap — judge score × variant

Rows are the 20 examples (sorted by true score). Columns are the 8 variants. Cell
color = judge score. The ground-truth column (leftmost) is for comparison.

Watch for **column-wise systematic shifts** — a variant that consistently scores lower
or higher across all rows. That's exactly what a semantic reframing does.

In [ ]:
sorted_idx = sorted(range(len(eval_data)), key=lambda i: eval_data[i]["true_score"])
true_col = np.array([[eval_data[i]["true_score"] for i in sorted_idx]]).T  # (n, 1)
judge_mat = np.array([[r["results"][i]["judge"] for r in all_results] for i in sorted_idx])  # (n, v)
full = np.hstack([true_col, judge_mat])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(full, cmap="RdYlGn", vmin=1, vmax=5, aspect="auto")
col_labels = ["TRUE"] + [r["name"].replace(" ", "\n", 1) for r in all_results]
ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=0, fontsize=8)
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([f"ex{eval_data[i]['id']} (t={eval_data[i]['true_score']})" for i in sorted_idx], fontsize=8)
for i in range(full.shape[0]):
    for j in range(full.shape[1]):
        ax.text(j, i, str(int(full[i, j])), ha="center", va="center", color="black", fontsize=8)
ax.axvline(0.5, color="black", linewidth=2)  # separate TRUE column
ax.set_title("Judge score per example × variant (leftmost = ground truth)")
plt.colorbar(im, ax=ax, label="score")
plt.tight_layout()
plt.show()

### d) Score-distribution histograms per variant

How each variant *allocates* its 20 scores across the 1–5 bins. Shape differences =
genuinely different instruments.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5.5), sharey=True)
for ax, r in zip(axes.flat, all_results):
    scores = [x["judge"] for x in r["results"]]
    ax.hist(scores, bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
            rwidth=0.85, color=color_map[r["kind"]], alpha=0.85)
    ax.set_title(f"{r['name']}\nmean={r['mean_score']:.2f}", fontsize=9)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xlim(0.5, 5.5)
    ax.grid(alpha=0.3, axis="y")
for ax in axes[:, 0]:
    ax.set_ylabel("count")
for ax in axes[-1, :]:
    ax.set_xlabel("score")
plt.tight_layout()
plt.show()

## Step 7 — Takeaways

### Main finding

- **Presentation perturbations barely moved anything** (v1 Δ ≈ 0.1 on a 1–5 scale).
  On modern models, formatting robustness is largely solved.
- **Semantic perturbations moved the mean score by up to 0.75 points** — nearly a
  full scale point, from a rubric rewording that no reviewer would flag.

### The v5 "accidental improvement" trap

v5 (customer perspective) produced the **best** exact-match score (55%, +25pp over
baseline) — not because the judge got better, but because renaming the instrument
happened to align it with the ground-truth distribution. If you weren't paying
attention, you'd ship this as an iteration win. **You didn't improve the judge;
you swapped the judge.**

### Per-item range is misleading on its own

0/20 examples had a score range ≥ 2 across variants — *per item* the scores look
stable. But the shifts are **systematic across many items**, so the aggregate mean
moves by up to 0.75 points. Always look at per-variant means, not just per-item variance.

### How to spot this in practice

1. **If you changed the rubric, you changed the metric.** Re-baseline before
   comparing iterations.
2. **Pin your rubric exactly.** Store it in version control. Any edit = a new
   version of the metric, not a minor tweak.
3. **Sensitivity-test the rubric, not just the prompt.** Generate paraphrases
   of your rubric before shipping and measure the spread of mean scores.
4. **Prefer stable rubrics.** If a paraphrase moves the mean by ≥ 0.3 points on
   a 1–5 scale, your rubric is fragile — tighten the wording until the mean stops
   moving.

### The real lesson

Modern LLMs aren't "brittle to punctuation." They are **exquisitely sensitive to
what you ask them to measure**. A rubric is an instrument; its wording is the
calibration. Treat edits to rubric wording with the same care as edits to a sensor's
firmware.